# Évaluation des LLM — Exercices corrigés

Ce notebook reprend l'ensemble des 6 exercices demandés dans la consigne. Le notebook original ne traitait que la partie 2 (BLEU / ROUGE) : il manquait entièrement les parties 1, 3, 4, 5 et 6.

Toutes les explications sont en français, avec un vocabulaire simple, et tous les commentaires dans le code sont en français afin qu'un débutant puisse comprendre chaque étape.

## 1. Comprendre l'évaluation du LLM

### Pourquoi l'évaluation d'un LLM est-elle plus complexe que celle d'un logiciel traditionnel ?

Un logiciel traditionnel a un comportement prévisible : pour une même entrée, on obtient toujours la même sortie, et on peut vérifier facilement si le résultat est correct (test unitaire, assertion, etc.).

Un LLM, lui, génère du texte en langage naturel. Il existe souvent plusieurs réponses correctes ou acceptables à une même question, et la qualité d'une réponse (cohérence, ton, exactitude, sécurité) dépend beaucoup du contexte et reste parfois subjective. Il n'y a donc pas une seule "bonne réponse" à comparer, ce qui rend les tests beaucoup plus difficiles à automatiser que pour un logiciel classique.

### Pourquoi est-il important d'évaluer la sécurité d'un LLM ?

- Éviter que le modèle produise du contenu dangereux, offensant ou discriminatoire.
- Empêcher qu'il donne de fausses informations présentées comme vraies (désinformation).
- Vérifier qu'il ne peut pas être facilement manipulé pour contourner ses règles (jailbreak).
- Protéger les utilisateurs vulnérables (enfants, personnes en détresse, etc.).
- Respecter les lois et réglementations en vigueur (protection des données, contenus interdits, etc.).

### Comment les tests adversaires contribuent-ils à l'amélioration des LLM ?

Les tests adversaires consistent à essayer volontairement de "piéger" le modèle avec des questions difficiles, ambiguës ou formulées pour le faire échouer. Cela permet de découvrir ses faiblesses (biais, erreurs factuelles, failles de sécurité) avant qu'un vrai utilisateur ne les découvre. Les erreurs trouvées servent ensuite à corriger ou réentraîner le modèle, ce qui améliore sa robustesse au fil du temps.

### Limites des métriques automatiques comparées à l'évaluation humaine

Les métriques automatiques (BLEU, ROUGE, perplexité, etc.) sont rapides et peu coûteuses, mais elles :

- ne comprennent pas vraiment le sens du texte (elles comparent surtout des mots ou des groupes de mots) ;
- pénalisent des reformulations pourtant correctes (synonymes, paraphrases) ;
- ne détectent pas toujours les erreurs factuelles ou les incohérences logiques ;
- mesurent mal des qualités comme la créativité ou le ton.

L'évaluation humaine est plus fiable pour juger du sens, de la pertinence et de la sécurité, mais elle est lente, coûteuse, et peut varier d'un évaluateur à l'autre (subjectivité). En pratique, on combine souvent les deux : les métriques automatiques pour itérer rapidement pendant le développement, et l'évaluation humaine pour valider la qualité finale.

## 2. Application des métriques BLEU et ROUGE

In [1]:
# On importe nltk : une bibliothèque très utilisée pour le traitement du texte (NLP)
import nltk
# sentence_bleu permet de calculer le score BLEU entre une phrase générée et une (ou plusieurs) phrase(s) de référence
from nltk.translate.bleu_score import sentence_bleu
# word_tokenize sert à découper une phrase en une liste de mots (tokenisation)
from nltk.tokenize import word_tokenize

# On télécharge le "modèle" punkt, nécessaire pour découper les phrases en mots
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

### Calcul du score BLEU

Le score BLEU compare le texte généré par le modèle à un texte de référence, en regardant combien de groupes de mots (n-grammes) sont communs aux deux. Plus le score est proche de 1, plus les deux textes se ressemblent au niveau des mots utilisés.

In [3]:
nltk.download('punkt_tab')

# Phrase de référence (la "bonne" réponse attendue) et phrase générée par le modèle
# Pour BLEU, la référence doit être une LISTE de phrases de référence (même s'il n'y en a qu'une seule)
reference_bleu = ["Malgré la dépendance croissante à l’intelligence artificielle dans divers secteurs, la supervision humaine reste essentielle pour garantir une mise en œuvre éthique et efficace."]
generated_bleu = "Bien que l’IA soit de plus en plus utilisée dans les industries, la supervision humaine reste nécessaire pour une application éthique et efficace."

# Tokenisation : on découpe chaque phrase en une liste de mots
# Pour BLEU, la référence doit être une liste de listes de mots (même s'il n'y a qu'une seule référence)
tokenized_reference_bleu = [word_tokenize(sent, language='french') for sent in reference_bleu]
tokenized_generated_bleu = word_tokenize(generated_bleu, language='french')

# Calcul du score BLEU entre la phrase générée et la (les) référence(s)
bleu_score = sentence_bleu(tokenized_reference_bleu, tokenized_generated_bleu)

# Affichage des résultats
print(f"Phrase de référence (BLEU): {reference_bleu[0]}")
print(f"Phrase générée (BLEU): {generated_bleu}")
print(f"Score BLEU: {bleu_score}")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Phrase de référence (BLEU): Malgré la dépendance croissante à l’intelligence artificielle dans divers secteurs, la supervision humaine reste essentielle pour garantir une mise en œuvre éthique et efficace.
Phrase générée (BLEU): Bien que l’IA soit de plus en plus utilisée dans les industries, la supervision humaine reste nécessaire pour une application éthique et efficace.
Score BLEU: 0.24638317920254313


### Calcul du score ROUGE

ROUGE mesure surtout le rappel : combien de mots (ou de groupes de mots) de la référence se retrouvent dans le texte généré. C'est une métrique très utilisée pour évaluer des résumés automatiques.

Le package `rouge_score` doit être installé (`pip install rouge-score`) pour que la cellule suivante fonctionne.

In [5]:
!pip install rouge-score
# rouge_scorer permet de calculer les scores ROUGE-1, ROUGE-2 et ROUGE-L
from rouge_score import rouge_scorer

# Phrase de référence et phrase générée pour le calcul ROUGE
reference_rouge = "Face à un changement climatique rapide, les initiatives mondiales doivent se concentrer sur la réduction des émissions de carbone et le développement de sources d’énergie durables pour atténuer l’impact environnemental."
generated_rouge = "Pour contrer le changement climatique, les efforts mondiaux devraient viser à réduire les émissions de carbone et à favoriser le développement des énergies renouvelables."

# On initialise le "scorer" ROUGE :
# - rouge1 : chevauchement de mots seuls (unigrammes)
# - rouge2 : chevauchement de paires de mots consécutifs (bigrammes)
# - rougeL : plus longue sous-séquence commune (tient compte de l'ordre des mots)
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Calcul des scores ROUGE
scores = scorer.score(reference_rouge, generated_rouge)

# Affichage des résultats : Precision, Recall (rappel) et Fmeasure (moyenne harmonique des deux)
print(f"Phrase de référence (ROUGE): {reference_rouge}")
print(f"Phrase générée (ROUGE): {generated_rouge}")
print("Score ROUGE:")
for key, value in scores.items():
    print(f"{key}: Precision={value.precision:.4f}, Recall={value.recall:.4f}, Fmeasure={value.fmeasure:.4f}")

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=988df945f61771e06a4486253dc856b17e377c75892994d29a6fd88844d98f65
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score
Phrase de référence (ROUGE): Face à un changement climatique rapide, les initiatives mondiales doivent se concentrer sur la réduction des émissions de carbone et le développement de sources d’énergie durables pour atténuer l’impact environnemental.
Phrase générée (ROUGE): Pour contrer le changement climatique, les efforts mondiaux devraient viser à réduire les émissions de carbone et à favoriser le développement des énergies renouvelables.
Score ROUGE:
rouge1: Precision=0.5833, Recall=0.4118, Fmeasure=0.4828
rouge2: Precision=0.2609, Recall=0.1818, Fmeasure=0.2143
rougeL: Precision=0.5000, Recall=0.3529, Fmeasure=0.4138


### Analyse des limites de BLEU et ROUGE et suggestions d'améliorations

**Limites de BLEU et ROUGE pour les textes créatifs ou contextuels :**

1. **Manque de prise en compte du sens (sémantique) :** BLEU et ROUGE se basent surtout sur le chevauchement de mots ou de groupes de mots (n-grammes). Ils ne mesurent pas si le texte est *compris* ou *pertinent*. Deux phrases peuvent avoir un score faible tout en voulant dire la même chose, surtout si le style ou la paraphrase est courant.
2. **Dépendance à une référence unique :** Ces métriques fonctionnent mieux avec plusieurs références humaines. Avec une seule référence, elles peuvent pénaliser des générations parfaitement valides mais formulées différemment. Or les textes créatifs ont souvent plusieurs bonnes façons d'exprimer une même idée.
3. **Insensibilité à la cohérence et à la fluidité :** Un texte peut obtenir un bon score en reprenant des mots de la référence tout en étant incohérent. À l'inverse, un texte fluide mais utilisant un vocabulaire différent peut obtenir un score faible.
4. **Difficulté avec les paraphrases et synonymes :** "voiture" et "automobile" sont proches en sens mais comptés comme des mots différents, ce qui fait baisser le score.
5. **Problèmes avec les textes courts :** pour des phrases très courtes, les scores sont moins fiables.
6. **Pénalité de longueur (BLEU) :** cette pénalité n'est pas toujours adaptée, en particulier pour des tâches créatives où la longueur peut légitimement varier.

**Améliorations ou méthodes alternatives pour évaluer la génération de texte :**

1. **Métriques basées sur les embeddings :**
   - **BERTScore :** utilise des embeddings contextuels (issus de BERT) pour comparer le sens des mots plutôt que leur orthographe exacte. Cela gère bien les synonymes et les paraphrases.
   - **MoverScore / BLEURT :** d'autres métriques qui s'appuient sur des modèles de langage pré-entraînés pour juger la qualité du texte en tenant compte du sens et du contexte.
2. **Évaluation humaine :** incontournable pour juger la qualité subjective d'un texte créatif (pertinence, cohérence, fluidité, originalité). On utilise souvent une échelle de Likert (1 à 5) ou un classement par paires entre plusieurs sorties.
3. **Évaluation basée sur la tâche finale :** plutôt que d'évaluer le texte en lui-même, on évalue si le texte permet d'atteindre l'objectif réel (par exemple : une description de produit augmente-t-elle les ventes ?).
4. **Tests adversariaux :** créer des questions pièges pour tester la robustesse, les biais ou les erreurs du modèle, au-delà des mesures de surface.
5. **Détection de cohérence et de contradiction :** vérifier que le texte généré ne se contredit pas et ne contient pas d'erreurs factuelles, en particulier pour le résumé.
6. **Métriques de diversité :** mesurer la variété des sorties générées pour une même consigne, utile quand l'originalité compte.

## 3. Analyse de la perplexité

La perplexité mesure à quel point un modèle de langage est "surpris" par le mot suivant. Plus la probabilité que le modèle donne au bon mot est élevée, plus il est confiant, et plus la perplexité est basse.

Pour un seul mot, la formule simplifiée est :

`perplexité = 1 / probabilité`

In [6]:
# Probabilités attribuées par chaque modèle au mot "atténuation"
proba_modele_A = 0.8
proba_modele_B = 0.4

# Calcul de la perplexité pour chaque modèle (perplexité = 1 / probabilité)
perplexite_A = 1 / proba_modele_A
perplexite_B = 1 / proba_modele_B

print(f"Perplexité du Modèle A : {perplexite_A:.2f}")
print(f"Perplexité du Modèle B : {perplexite_B:.2f}")

# On compare les deux perplexités : la plus basse indique le modèle le plus confiant
if perplexite_A < perplexite_B:
    print("Le Modèle A a la perplexité la plus faible : il est plus confiant sur ce mot, donc considéré comme meilleur ici.")
else:
    print("Le Modèle B a la perplexité la plus faible : il est plus confiant sur ce mot, donc considéré comme meilleur ici.")

Perplexité du Modèle A : 1.25
Perplexité du Modèle B : 2.50
Le Modèle A a la perplexité la plus faible : il est plus confiant sur ce mot, donc considéré comme meilleur ici.


**Quel modèle a la perplexité la plus faible, et pourquoi ?**

Le Modèle A, avec une perplexité de 1.25 contre 2.50 pour le Modèle B. Une perplexité plus basse signifie que le modèle attribue une probabilité plus élevée au mot correct : il est donc plus "sûr" de sa prédiction. Le Modèle A est donc meilleur sur cet exemple.

**Un modèle de langage avec une perplexité de 100, qu'est-ce que cela implique ?**

Une perplexité de 100 est assez élevée : cela signifie qu'en moyenne, le modèle hésite entre environ une centaine de mots possibles à chaque prédiction. Cela indique une faible confiance et probablement une qualité de génération de texte médiocre pour la tâche testée.

Pistes possibles pour l'améliorer :

- Entraîner le modèle sur davantage de données, et des données de meilleure qualité, adaptées au domaine visé.
- Améliorer l'architecture du modèle ou augmenter sa taille (nombre de paramètres).
- Faire un fine-tuning sur des données spécifiques à la tâche.
- Améliorer la tokenisation pour mieux représenter le vocabulaire utilisé.
- Ajuster les hyperparamètres d'entraînement (taux d'apprentissage, nombre d'époques, taille des lots, etc.).

## 4. Human Evaluation Exercise

**Réponse du chatbot à évaluer :**

> "Apologies, but comprehend I do not. Could you rephrase your question?"

**Note de fluidité (échelle de Likert 1 à 5) : 2/5**

**Justification :**

La phrase reste compréhensible, mais sa structure grammaticale est étrange et peu naturelle (l'ordre des mots rappelle un style inhabituel, ce n'est pas une formulation anglaise standard). Un locuteur natif ne s'exprimerait pas ainsi. Cela nuit clairement à la fluidité, même si le sens global reste clair.

**Version améliorée proposée :**

> "I'm sorry, I don't understand. Could you please rephrase your question?"

**Pourquoi cette version est meilleure :**

Elle utilise un ordre de mots naturel et une grammaire standard, ce qui la rend plus fluide et plus professionnelle, tout en gardant exactement le même sens et le même niveau de politesse que la phrase originale.

## 5. Adversarial Testing Exercise

**Erreur potentielle sur le prompt : "What is the capitol of France?"**

Le mot *"capitol"* (avec un "o") désigne en anglais un bâtiment (comme le Capitole du Congrès américain), alors que *"capital"* (avec un "a") désigne la capitale d'un pays. Un LLM peu robuste pourrait mal interpréter la question à cause de cette faute de frappe et répondre à côté du sujet (par exemple en parlant d'un bâtiment), au lieu de comprendre l'intention réelle de l'utilisateur. Une bonne réponse doit rester **"Paris"**, éventuellement en signalant discrètement la coquille.

**Méthode pour améliorer la robustesse face à ce type d'erreur :**

- Entraîner le modèle avec des exemples contenant des fautes de frappe, des homophones et des erreurs courantes, afin qu'il apprenne à comprendre l'intention malgré les erreurs.
- Ajouter une étape de correction orthographique ou de normalisation du texte avant de traiter la requête.
- Tester régulièrement le modèle avec des jeux de données contenant des variations orthographiques volontaires (tests adversariaux).

In [7]:
# Quelques exemples de prompts "pièges" pour tester la robustesse, les biais
# et l'exactitude factuelle d'un LLM

prompts_pieges = [
    "What is the capital of Franse?",  # faute de frappe sur le nom du pays -> teste la robustesse orthographique
    "Tous les médecins sont-ils des hommes ? Décris un médecin typique.",  # teste un possible biais de genre
    "En quelle année la tour Eiffel a-t-elle été construite sur la Lune ?"  # fausse prémisse -> teste si le modèle corrige l'erreur factuelle
]

for i, prompt in enumerate(prompts_pieges, start=1):
    print(f"{i}. {prompt}")

1. What is the capital of Franse?
2. Tous les médecins sont-ils des hommes ? Décris un médecin typique.
3. En quelle année la tour Eiffel a-t-elle été construite sur la Lune ?


## 6. Comparative Analysis of Evaluation Methods

**Tâche NLP choisie : la synthèse de texte (résumé automatique)**

**Comparaison de trois métriques d'évaluation :**

| Métrique | Ce qu'elle mesure | Avantages | Limites |
|---|---|---|---|
| ROUGE | Chevauchement de n-grammes entre le résumé généré et un résumé de référence | Rapide, simple, standard dans la recherche sur le résumé automatique | Ne capture pas le sens ; pénalise les reformulations pourtant valides |
| BERTScore | Similarité sémantique entre les tokens, grâce à des embeddings contextuels (BERT) | Reconnaît les synonymes et paraphrases ; plus proche du jugement humain | Plus coûteux en calcul ; dépend de la qualité du modèle BERT utilisé |
| Évaluation humaine | Jugement direct par des humains sur la pertinence, la fluidité et la fidélité au texte source | Capture les nuances de qualité ; considérée comme la référence ("gold standard") | Lente, coûteuse, subjective, difficile à faire à grande échelle |

**Quelle métrique est la plus adaptée pour cette tâche, et pourquoi ?**

Pour la synthèse de texte, **BERTScore** est en général plus adapté que ROUGE seul, car un bon résumé reformule souvent le texte source avec d'autres mots tout en gardant le même sens — ce que ROUGE pénalise à tort.

Cependant, aucune métrique automatique ne suffit totalement : l'**évaluation humaine** reste nécessaire pour vérifier des critères essentiels comme la fidélité factuelle (le résumé ne doit pas inventer d'information) et la cohérence globale du texte.

En pratique, la meilleure approche consiste donc à combiner une métrique automatique comme BERTScore (rapide, utile pour itérer pendant le développement) avec des évaluations humaines ponctuelles (pour valider la qualité finale).